In [1]:
from pathlib import Path
import sys
import numpy as np

_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(p for p in _candidates if (p / "src" / "constellation_denoiser").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from constellation_denoiser import *

## Load and split 70/15/15

In [2]:
tx, rx = load_channel_data("Symbols_1m_1ch_PR_1.csv")
(tx_train, rx_train), (tx_val, rx_val), (tx_test, rx_test) = split(tx, rx)

## Windows for linear equalizer

In [3]:
window = 5

def make_windows(rx, tx, window):
    X = np.lib.stride_tricks.sliding_window_view(rx, window)
    y = tx[window // 2 : -(window // 2)]
    ones = np.ones((X.shape[0], 1), dtype=X.dtype)
    return np.hstack([X, ones]), y

X_train, Y_train = make_windows(rx_train, tx_train, window)
X_val,   Y_val   = make_windows(rx_val, tx_val, window)

## Fit on train

In [4]:
w, *_ = np.linalg.lstsq(X_train, Y_train, rcond=None)

## Baseline BER (same val split)

In [5]:
ber_raw = calculate_ber(symbols_to_bits(rx_val), symbols_to_bits(tx_val))
print(f"BER raw = {ber_raw:.6f}")

BER raw = 0.008747


## FFE BER (val)

In [6]:
y_pred = X_val @ w
ber_ffe = calculate_ber(symbols_to_bits(y_pred), symbols_to_bits(Y_val))
print(f"BER FFE = {ber_ffe:.6f}")

BER FFE = 0.009678
